# Time series processing

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import json
import os
import shutil

import numpy as np
import xarray as xr


In [ ]:
from sweat.api import read_ts_input_data, run_timeseries
from sweat.timeseries.config import check_config_timeseries
from sweat.timeseries.io_handler import write_timeseries

In [ ]:
min_date_str = "2023-03-01"
max_date_str = "2023-03-07"

In [ ]:
et_single_date_dir = (
    "/home/paul-rohel/Bureau/timeseries/et-dataset/timeseries/et_copy/"
)
radiation_dir = (
    "/home/paul-rohel/Bureau/timeseries/et-dataset/timeseries/daily_radiation/"
)
et_time_series_dir = (
    "/home/paul-rohel/Bureau/timeseries/sweat/et_time_series_dir/"
)
json_dir = "/home/paul-rohel/Bureau/timeseries/sweat/json_dir"


In [ ]:
def create_file(et_time_series_dir: str, config_dict: dict) -> None:
    config = check_config_timeseries(config_dict)
    et_ts, radiation_ts, et_sd, dem = read_ts_input_data(config_dict["input"])
    updated_ts = run_timeseries(
        et_ts, radiation_ts, et_sd, dem, config["params"]
    )
    write_timeseries(
        updated_ts, root_name="et_time_series", directory=et_time_series_dir
    )

In [ ]:
def create_copy(
    et_single_date_path: str,
    et_time_series_dir: str,
    et_time_series_path: str,
    et_single_date_filename: str,
) -> None:
    shutil.copy2(et_single_date_path, et_time_series_dir)
    to_rename = os.path.join(et_time_series_dir, et_single_date_filename)
    os.rename(to_rename, et_time_series_path)

In [ ]:
def create_config(
    t_start: str,
    t_end: str,
    et_single_date_dir: str,
    radiation_dir: str,
    et_time_series_dir: str,
    window: int,
    shift: int,
    json_dir: str | None = None,
) -> None:
    time = xr.date_range(t_start, freq="1D", end=t_end)
    cursor = 1
    cursor_prec = 0
    os.makedirs(json_dir, exist_ok=True)
    os.makedirs(et_time_series_dir, exist_ok=True)
    config_dict = {
        "input": {
            "dates": [],
            "et_time_series": [],
            "radiation": [],
            "et_single_date": [],
        },
        "output": {"path": et_time_series_dir},
        "params": {},
    }
    while cursor_prec < len(time):
        while len(config_dict["input"]["dates"]) >= window:
            date = config_dict["input"]["dates"].pop(0)
            config_dict["input"]["radiation"].pop(0)
            config_dict["input"]["et_time_series"].pop(0)
            to_remove = os.path.join(
                et_single_date_dir,
                f"et_single_date_{dt.datetime.strptime(date, ('%Y-%m-%d')).astimezone(dt.UTC).strftime('%Y%m%d')}.tif",
            )
            if to_remove in config_dict["input"]["et_single_date"]:
                config_dict["input"]["et_single_date"].remove(to_remove)
        for i in range(cursor_prec, cursor):
            if i < len(time):
                config_dict["input"]["dates"].append(
                    time[i].strftime("%Y-%m-%d")
                )
                radiation_path = os.path.join(
                    radiation_dir, f"radiation_{time[i].strftime('%Y%m%d')}.tif"
                )
                config_dict["input"]["radiation"].append(radiation_path)
                et_single_date_filename = (
                    f"et_single_date_{time[i].strftime('%Y%m%d')}.tif"
                )
                if et_single_date_filename in os.listdir(et_single_date_dir):
                    et_single_date_path = os.path.join(
                        et_single_date_dir, et_single_date_filename
                    )
                    config_dict["input"]["et_single_date"].append(
                        et_single_date_path
                    )
                et_time_series_filename = (
                    f"et_time_series_{time[i].strftime('%Y%m%d')}.tif"
                )
                et_time_series_path = os.path.join(
                    et_time_series_dir,
                    et_time_series_filename,
                )
                create_file(et_time_series_dir, config_dict)
                config_dict["input"]["et_time_series"].append(
                    et_time_series_path
                )

        with open(f"{json_dir}/config_{cursor_prec}.json", "w") as f:
            json.dump(config_dict, f, indent=4)
        cursor_prec = cursor
        cursor = cursor + shift

In [ ]:
shutil.rmtree(json_dir, ignore_errors=True)
shutil.rmtree(et_time_series_dir, ignore_errors=True)
create_config(
    min_date_str,
    max_date_str,
    et_single_date_dir,
    radiation_dir,
    et_time_series_dir,
    4,
    1,
    json_dir,
)


In [ ]:
json_path = os.path.join(json_dir, "config_0.json")
with open(json_path) as json_file:
    data = json.load(json_file)


In [ ]:
config = check_config_timeseries(data)

In [ ]:
data["input"]["et_time_series"].pop(0)

In [ ]:
data

In [ ]:
et_ts, radiation_ts, et_sd, dem = read_ts_input_data(data["input"])


In [ ]:
radiation_ts["daily_radiation"] = radiation_ts["daily_radiation"] * np.nan

In [ ]:
et_ts

In [ ]:
updated_ts = run_timeseries(et_ts, radiation_ts, et_sd, dem, config["params"])


In [ ]:
updated_ts

In [ ]:
updated_ts.et[0, :, :].plot()